# Information Health — one-click Colab demo

Runs the **FastAPI engine** + a **production build** of the Next.js app inside Colab and
opens a public URL.

- A *production* build is served on purpose: `next dev` injects CSS via JS and uses a
  hot-reload websocket that misbehaves behind a tunnel (blank / unstyled page). Production
  ships a real stylesheet + static chunks, so it renders reliably.
- The onboarding → **Initial Information Health Estimate** flow needs no credentials.
- Optional cells: **real publisher names** (Qbias profile) and **Google sign-in**.

Run the cells top to bottom (Runtime → *Run all* also works).

> If your repo is **private**, paste a GitHub token (read access) in step 1.


In [ ]:
#@title 1 · Clone the repo + install Node 20
REPO   = "greenwichg/random_walks_with_erasure"  #@param {type:"string"}
BRANCH = "claude/sleepy-gates-oecof1"             #@param {type:"string"}
GITHUB_TOKEN = ""  #@param {type:"string"}   # only needed if the repo is private

auth = f"{GITHUB_TOKEN}@" if GITHUB_TOKEN else ""
!rm -rf app && git clone --depth 1 --branch {BRANCH} https://{auth}github.com/{REPO}.git app
%cd app
# Colab ships an old Node; install a modern one for Next.js 14.
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash - >/dev/null 2>&1
!sudo apt-get install -y nodejs >/dev/null 2>&1
!node -v ; npm -v

In [ ]:
#@title 2 · Start the FastAPI engine (synthetic corpus, no external data)
import subprocess, time, urllib.request

!pip install -q -e ".[serve]"

def wait(url, n=180):
    for _ in range(n):
        try:
            if urllib.request.urlopen(url, timeout=2).status == 200: return True
        except Exception: time.sleep(1)
    return False

engine = subprocess.Popen(["python", "examples/api_fastapi.py"],
                          stdout=open("engine.log", "w"), stderr=subprocess.STDOUT)
print("engine:", "UP" if wait("http://127.0.0.1:8000/api/health") else "FAILED — see engine.log")

In [ ]:
#@title 3 · Build the web app (production) + start it, pointed at the engine
import secrets
# Production is served (dev mode's HMR + JS-injected CSS break behind a tunnel).
# Prod also disables the mock fallback, so it uses the real engine from step 2.
open("web/.env.local", "w").write(
    "RWE_BACKEND_URL=http://127.0.0.1:8000\n"
    f"NEXTAUTH_SECRET={secrets.token_urlsafe(32)}\n")
!cd web && npm install --no-audit --no-fund --loglevel=error
!cd web && npm run build
web = subprocess.Popen(["npm", "start"], cwd="web",
                       stdout=open("web.log", "w"), stderr=subprocess.STDOUT)
print("web:", "UP" if wait("http://127.0.0.1:3000/onboarding") else "still starting — check web.log")

In [ ]:
#@title 4 · Open it — public URL via a Cloudflare quick tunnel
import re, time, subprocess
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
cf = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:3000", "--no-autoupdate"],
                      stdout=open("cf.log", "w"), stderr=subprocess.STDOUT)
PUBLIC_URL = None
for _ in range(40):
    time.sleep(1)
    try:
        m = re.search(r"https://[-\w.]+\.trycloudflare\.com", open("cf.log").read())
        if m: PUBLIC_URL = m.group(0); break
    except Exception: pass
print("\n👉  Open:", PUBLIC_URL or "no URL yet — re-run this cell or check cf.log")
print("    (it redirects to /onboarding — value → pick publishers → Initial Estimate)")

## Optional — real publisher names (Qbias)

The default `synthetic` profile shows generated outlet names ("Outlet N") with real
leans. To show **real publishers** (NYT, Fox, WSJ, NPR…) with real AllSides gold lean,
switch the engine to the **Qbias** profile below — configuration only, no app change.
Run it any time after step 3, then reload the app.


In [ ]:
#@title (optional) · Real publisher names — switch the engine to the Qbias AllSides catalog
import subprocess, time, os
# Config only: real outlets + real AllSides gold lean (~14 MB download).
!wget -q "https://raw.githubusercontent.com/irgroup/Qbias/main/allsides_balanced_news_headlines-texts.csv" -O qbias.csv
print("csv size:", os.path.getsize("qbias.csv"), "bytes")   # ~13.7M expected; ~0 means the download failed
try: engine.terminate(); time.sleep(2)
except Exception: pass
env = {**os.environ, "RWE_PROFILE": "qbias", "RWE_QBIAS": "qbias.csv",
       "RWE_MAX_ITEMS": "1200", "RWE_N_USERS": "300"}
engine = subprocess.Popen(["python", "examples/api_fastapi.py"], env=env,
                          stdout=open("engine.log", "w"), stderr=subprocess.STDOUT)
print("engine (qbias):", "UP" if wait("http://127.0.0.1:8000/api/health", 240) else "check engine.log")
print("→ reload the app — the onboarding outlet chips now show real publishers")

## Optional — enable Google sign-in

The onboarding + estimate work without this. To sign in and reach the dashboard:

1. In **Google Cloud Console → Credentials**, create an *OAuth client ID → Web application*.
2. Run the cell below with your client id/secret and the **public URL from step 4**.
3. It prints a redirect URI — add it to that OAuth client, then reload the app and sign in.

> The tunnel URL changes each session, so you re-add the redirect URI each run.


In [ ]:
#@title (optional) 5 · Configure Google OAuth, then restart the web server
CLIENT_ID     = ""  #@param {type:"string"}
CLIENT_SECRET = ""  #@param {type:"string"}
import secrets, time, subprocess
assert PUBLIC_URL, "run step 4 first to get the public URL"
open("web/.env.local", "w").write(
    "RWE_BACKEND_URL=http://127.0.0.1:8000\n"
    f"GOOGLE_CLIENT_ID={CLIENT_ID}\n"
    f"GOOGLE_CLIENT_SECRET={CLIENT_SECRET}\n"
    f"NEXTAUTH_SECRET={secrets.token_urlsafe(32)}\n"
    f"NEXTAUTH_URL={PUBLIC_URL}\n")
# server-side env is read when the server starts, so a restart is enough (no rebuild).
web.terminate(); time.sleep(2)
web = subprocess.Popen(["npm", "start"], cwd="web",
                       stdout=open("web.log", "w"), stderr=subprocess.STDOUT)
print("Add this redirect URI to your Google OAuth client, then reload the app:")
print(" ", PUBLIC_URL + "/api/auth/callback/google")
print("web restarting:", "UP" if wait("http://127.0.0.1:3000/onboarding") else "check web.log")